In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pickle
import os

print("1. Loading Data and Training Model...")
# 1. LOAD DATA & TRAIN MODEL
df = pd.read_csv('../data/final/master_training_data.csv')

# Handle missing data
df['home_ea_score'] = df['home_ea_score'].fillna(60)
df['away_ea_score'] = df['away_ea_score'].fillna(60)
df['home_tm_value'] = df['home_tm_value'].fillna(1000000)
df['away_tm_value'] = df['away_tm_value'].fillna(1000000)
df['home_fifa_rank'] = df['home_fifa_rank'].fillna(150)
df['away_fifa_rank'] = df['away_fifa_rank'].fillna(150)

# Reshape and train
home_data = pd.DataFrame({'team': df['home_team'], 'opponent': df['away_team'], 'goals': df['home_score'], 'is_home_advantage': np.where(df['neutral'] == True, 0, 1), 'rank_diff': df['away_fifa_rank'] - df['home_fifa_rank'], 'ea_diff': df['home_ea_score'] - df['away_ea_score'], 'tm_log_diff': np.log1p(df['home_tm_value']) - np.log1p(df['away_tm_value'])})
away_data = pd.DataFrame({'team': df['away_team'], 'opponent': df['home_team'], 'goals': df['away_score'], 'is_home_advantage': 0, 'rank_diff': df['home_fifa_rank'] - df['away_fifa_rank'], 'ea_diff': df['away_ea_score'] - df['home_ea_score'], 'tm_log_diff': np.log1p(df['away_tm_value']) - np.log1p(df['home_tm_value'])})
goal_data = pd.concat([home_data, away_data], ignore_index=True).dropna(subset=['goals'])

poisson_model = smf.glm(formula="goals ~ is_home_advantage + rank_diff + ea_diff + tm_log_diff", data=goal_data, family=sm.families.Poisson()).fit()

print("2. Building Latest Stats Dictionaries...")
# 2. BUILD LATEST STATS DICTIONARIES
df_ea = pd.read_csv('../data/processed/ea_squad_strength.csv')
df_tm = pd.read_csv('../data/processed/tm_squad_value.csv')
df_fifa = pd.read_csv('../data/processed/fifa_rankings.csv')

latest_ea = df_ea[df_ea['year'] == 2024].set_index('country')['ea_squad_score'].to_dict()
latest_tm = df_tm.sort_values('year').groupby('country').tail(1).set_index('country')['tm_squad_value_eur'].to_dict()
latest_fifa = df_fifa.sort_values('rank_date').groupby('country').tail(1).set_index('country')['fifa_rank'].to_dict()

latest_stats = {
    'ea': latest_ea,
    'tm': latest_tm,
    'fifa': latest_fifa
}

# Create a master list of all valid team names
team_names = sorted(list(latest_ea.keys()))

print("3. Exporting to Pickle files...")
# 3. EXPORT USING PICKLE
# Create the api/models directory if it doesn't exist
os.makedirs('../api/models', exist_ok=True)

# Export using Statsmodels' built-in safe saver
poisson_model.save('../api/models/poisson_model.pkl')

# Export the stats dictionary
with open('../api/models/latest_stats.pkl', 'wb') as f:
    pickle.dump(latest_stats, f)

# Export the team names list
with open('../api/models/team_names.pkl', 'wb') as f:
    pickle.dump(team_names, f)

print("✅ SUCCESS! All files saved in the 'api/models/' folder.")

1. Loading Data and Training Model...
2. Building Latest Stats Dictionaries...
3. Exporting to Pickle files...
✅ SUCCESS! All files saved in the 'api/models/' folder.
